This will be the clean version of the work with just the useful function

I need to do the doc string for every of this function

In [ ]:
import matplotlib.pyplot as plt
import numpy as np



#definition des variables

pi = np.pi  

In [ ]:
def effective_area_and_waist (alt, theta, waist_0, lambda_0) : 
    #permet d'obtenir l'effective area en m-2 et le waist en m avec 
    #les paramètres du télescope et de l'altitude étudiée
    w = waist_0*np.sqrt(1+ (lambda_0*alt/(pi*waist_0**2))**2)
    B = 2/pi*(lambda_0*alt/w)**2*np.exp(-2*(alt*np.tan(theta)/w)**2)
    return B,w

In [ ]:
#Mise en place de la méthode 1

import matplotlib.pyplot as plt
import numpy as np
import pycraf
from pycraf import conversions as cnv
from astropy import units as u
from scipy.interpolate import UnivariateSpline
from scipy.integrate import simpson
from scipy.integrate import trapezoid, cumulative_trapezoid

pi=np.pi

def alpha_specific_function(altitudes, frequency, Temperature, Pressure ,P_water) :
    """Entrée : altitudes : array (m), frequency : array (m)
    sortie : retourne le tableau de alpha spécifique (len(altitudes) lignes et len(frequency) colonnes)"""
    
    altitudes_km = altitudes * u.m       # maintenant c'est une Quantity en m
    altitudes_km = altitudes_km.to(u.km) # conversion en km
    frequency_GHz = frequency*10**-9 * u.GHz
    P_dry = Pressure - P_water
    

    #Calcul du coefficient d'absorbtion alpha à l'aide de pycraf
    alpha_specific = np.zeros((len(altitudes_km), len(frequency)))

    for i in range(len(altitudes_km)):
        
        # Atténuation pour toutes les fréquences à cette altitude
        alpha_dry_dB_km,alpha_wet_dB_km = pycraf.atm.atten_specific_annex1(frequency_GHz, P_dry[i], P_water[i], Temperature[i])
        # Conversion en m^-1
        alpha_tot_m = (alpha_dry_dB_km + alpha_wet_dB_km) * (np.log(10)/10) / 1000
        

        # Affectation à la ligne i
        alpha_specific[i, :] = alpha_tot_m.value   #en m-1
    return alpha_specific
    
    
def optical_depth_emission (altitudes, alpha_specific) :
    """Entrée : altitudes : array (m), frequency : array (m), alpha_specific : array (m-1) obtenable avec alpha_specific_function
    sortie : retourne le tableau de alpha spécifique (len(altitudes) lignes et len(frequency) colonnes)"""
    
    # altitudes : (N_alt,)
    # alpha_specific : (N_alt, N_freq)

    dz = np.diff(altitudes)  # (N_alt-1,)
    alpha_avg = 0.5 * (alpha_specific[1:, :] + alpha_specific[:-1, :])  # (N_alt-1, N_freq)

    # intégrales cumulées sur l'axe altitude
    tau = np.zeros_like(alpha_specific)
    tau[1:, :] = np.cumsum(alpha_avg * dz[:, None], axis=0)

    return tau


    


def contribution_effective_area (frequency, theta_b, altitudes, N) :
    """Entrée : frequency : array (Hz), theta_b (angle d'ouverture): array (rad), altitudes : array (m)
    theta_lim (borne d'intégration selon theta (entre 0 et pi/2) : array (rad), N (nombre de valeurs prises lors de l'intégration): scalar
    Il faut len (frequency) = len (theta_b)
    return : retourne le tableau de contribution effective area (len(altitudes) lignes et len(frequency) colonnes) """
    #definitions des variables pertinentes
    pi = np.pi     
    wavelength = 3.e8 / frequency #m
    w_0 = wavelength / (pi*theta_b) #m
    #Calcul de c_alt
    C_alt = np.zeros((len(altitudes),len(frequency))) #tableau pour enregistrer les contributions à chaque altitude et différentes fréquences
    altitudes_rel = altitudes - altitudes[0]
    

    for j in range(len(frequency)):
        """theta = np.linspace(0, theta_lim[j], N)"""  # (N,)
        theta= np.geomspace(0.000001, 1.57, N)
        wavelength_j = wavelength[j]
        w0_j = w_0[j]

        # On vectorise : altitudes[:,None] et theta[None,:]
        eff_area, _ = effective_area_and_waist(altitudes_rel[:, None], theta[None, :], w0_j, wavelength_j)  # shape (len(altitudes), N)

        f_values = (2 * pi * (altitudes[:, None] / wavelength_j)**2* np.abs(np.tan(theta))[None, :]* eff_area)

        # Intégrale Simpson sur theta, pour chaque altitude
        C_alt[:, j] = trapezoid(f_values, theta, axis=1)/altitudes**2  
    return C_alt


In [ ]:
#calcul avec les fonctions de la méthode 1

def Calcul_T_ant_1_el(frequency, theta_b, altitudes, N, Temperature, Pressure, P_water,elevation) :
    
    #definitions des variables pertinentes
    pi = np.pi 
    thet = 90 - elevation
    thet_rad = thet * pi/180
    m = 1/(np.cos(thet_rad) + 0.50572*(96.07995-thet)**(-1.6364))   
    wavelength = 3.e8 / frequency #m
    w_0 = wavelength / (pi*theta_b)
    
    altitudes_km = altitudes * u.m       # maintenant c'est une Quantity en m
    altitudes_km = altitudes_km.to(u.km) # conversion en km
    frequency_GHz = frequency*10**-9 * u.GHz

    #calcul de C_alt
    C_alt = contribution_effective_area(frequency, theta_b, altitudes, N)
   
            
    #Calcul de alpha

    alpha_specific= alpha_specific_function(altitudes, frequency, Temperature, Pressure, P_water)


    tau = optical_depth_emission (altitudes, alpha_specific)
    
    # on définit le terme que l'on va intégrer sur l'altitudes 
    C_tot = np.zeros((len(altitudes_km), len(frequency)))
    C_tot = C_alt * alpha_specific*m * Temperature[:, None] * np.exp(-tau*m)
    T_ant = trapezoid(C_tot, altitudes, axis=0)
        
    
    return T_ant

In [ ]:
#Mise en place de la méthode 2
import matplotlib.pyplot as plt
import numpy as np
import pycraf
from pycraf import conversions as cnv
from astropy import units as u

def Calcul_T_ant_2 (frequency, elevation = 90, obs_alt = 0) :
    """Entrée : Frequency : scalar or array (Hz), elevation (angle d'élevation): scalar (deg) 90 par défaut, 
    obs_alt (altitude de l'antenne par rapport au niveau de la mer) : scalar (km) 0 par défaut 
    renvoie un array avec les valeurs de T_ant2 pour chaque valeur de fréquency"""
     
    
    frequency_GHz = frequency*10**-9 * u.GHz

    elevation = elevation * u.deg #on regarde le zenith
    obs_alt = obs_alt * u.km # on se place au niveau de la mer
    atl_layer_cache = pycraf.atm.atm_layers(frequency_GHz, pycraf.atm.profile_standard , heights=None)


    T_ant2 = pycraf.atm.atten_slant_annex1(elevation, obs_alt, atl_layer_cache, do_tebb=True, t_bg= 2.73 * u.K, max_arc_length= 180 * u.deg, max_path_length= 100. * u.km) [2]
    
    return T_ant2

In [ ]:
def generate_Pwater_MC(P_water_ref, N_MC, rel_sigma=0.1, rng=None):
    """
    Génère un tableau (N_MC, ...) de P_water autour de P_water_ref
    en supposant une loi normale centrée avec écart-type = rel_sigma * P_water_ref.
    """
    rng = np.random.default_rng() if rng is None else rng

    P_water_ref = np.asarray(P_water_ref)
    sigma = rel_sigma * np.where(P_water_ref==0, 1.0, P_water_ref)

    # Tirages Gaussiens
    draws = rng.normal(loc=P_water_ref, scale=sigma, size=(N_MC,) + P_water_ref.shape)

    # Troncature à 0 pour éviter des valeurs négatives
    draws = np.maximum(draws, 0.0000000000001)

    return draws  # <--- maintenant c'est un seul array de shape (N_MC, *P_water_ref.shape)


In [ ]:
def generate_Pwater_MC_lognormal(P_water_ref, N_MC, rel_sigma, rng=None):
    """
    Génère un tableau (N_MC, ...) de P_water > 0 autour de P_water_ref
    en supposant une loi log-normale avec CV = rel_sigma (peut être un scalaire ou un array).
    
    Parameters
    ----------
    P_water_ref : array-like
        Profil de P_water de référence (>0), shape (n_alt,) ou similaire.
    N_MC : int
        Nombre de tirages Monte-Carlo.
    rel_sigma : float or array-like
        Incertitude relative (CV). Peut être un scalaire ou un array de même shape que P_water_ref.
    rng : np.random.Generator, optional
        Générateur pseudo-aléatoire NumPy.
    
    Returns
    -------
    draws : ndarray
        Tableau de shape (N_MC, *P_water_ref.shape) avec les tirages Monte-Carlo log-normaux.
    """
    rng = np.random.default_rng() if rng is None else rng

    P_water_ref = np.asarray(P_water_ref, dtype=float)
    rel_sigma = np.asarray(rel_sigma, dtype=float)

    # Coefficient de variation (évite divisions par 0)
    cv = np.maximum(rel_sigma, 1e-12)

    # Paramètres log-normaux
    sigma2_ln = np.log(1.0 + cv**2)
    sigma_ln = np.sqrt(sigma2_ln)
    mu_ln = np.log(np.where(P_water_ref > 0, P_water_ref, 1e-20)) - 0.5 * sigma2_ln

    # Tirages log-normaux
    draws = rng.lognormal(mean=mu_ln, sigma=sigma_ln,
                          size=(N_MC,) + P_water_ref.shape)

    return draws